**Extracting absolute counts for Representative Days**

In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell"
)

IN_FILE = BASE_DIR / "detector_data_CSV_new" / "all_intersections_common_days_long.csv"

OUT_DIR = BASE_DIR / "representative_day_routesampler_base_warmup15"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WARMUP_SECONDS = 15 * 60  # 15 min

SCENARIOS = [
    {
        "name": "weekday_morning",
        "date": "2026-03-10",
        "start_time": "07:45:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekday_evening",
        "date": "2026-03-23",
        "start_time": "15:45:00",
        "end_time": "17:00:00"
    },
    {
        "name": "weekend_morning",
        "date": "2026-03-14",
        "start_time": "07:45:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekend_evening",
        "date": "2026-03-22",
        "start_time": "15:45:00",
        "end_time": "17:00:00"
    }
]

# ============================================================
# HELPERS
# ============================================================

def hhmmss_to_seconds(t: str) -> int:
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(IN_FILE)

# robust timestamp handling
if "timestamp_local" in df.columns:
    df["timestamp_local"] = pd.to_datetime(df["timestamp_local"], errors="coerce")
elif "timestamp" in df.columns:
    df["timestamp_local"] = pd.to_datetime(df["timestamp"], errors="coerce")
else:
    raise ValueError("No timestamp column found. Expected 'timestamp_local' or 'timestamp'.")

if "timestamp_utc" in df.columns:
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce", utc=True)
else:
    df["timestamp_utc"] = pd.NaT

df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

if "time" not in df.columns:
    df["time"] = df["timestamp_local"].dt.strftime("%H:%M:%S")

df["count"] = pd.to_numeric(df.get("count", 0), errors="coerce").fillna(0)
df["occupancy"] = pd.to_numeric(df.get("occupancy", 0), errors="coerce").fillna(0)

# speed / velocity compatibility
if "speed" in df.columns:
    df["speed"] = pd.to_numeric(df["speed"], errors="coerce").fillna(0)
elif "velocity" in df.columns:
    df["speed"] = pd.to_numeric(df["velocity"], errors="coerce").fillna(0)
else:
    df["speed"] = 0.0

# create begin_s / end_s from timestamp if not present
if "begin_s" not in df.columns:
    df["begin_s"] = (
        df["timestamp_local"].dt.hour * 3600 +
        df["timestamp_local"].dt.minute * 60 +
        df["timestamp_local"].dt.second
    )

if "end_s" not in df.columns:
    # assumes 15-minute detector aggregation
    df["end_s"] = df["begin_s"] + 900

df["begin_s"] = pd.to_numeric(df["begin_s"], errors="coerce")
df["end_s"] = pd.to_numeric(df["end_s"], errors="coerce")

# add missing columns for compatibility
for col in ["edge_id", "movement", "count_type", "from_edge", "to_edge", "sumo_id", "approach"]:
    if col not in df.columns:
        df[col] = pd.NA

# ============================================================
# EXTRACT PER SCENARIO
# ============================================================

summary_rows = []

for sc in SCENARIOS:
    name = sc["name"]
    rep_day = sc["date"]
    start_time = sc["start_time"]
    end_time = sc["end_time"]

    scenario_start_s = hhmmss_to_seconds(start_time)
    scenario_end_s = hhmmss_to_seconds(end_time)
    analysis_start_s = scenario_start_s + WARMUP_SECONDS
    analysis_end_s = scenario_end_s

    temp = df[
        (df["date"] == rep_day) &
        (df["time"] >= start_time) &
        (df["time"] < end_time)
    ].copy()

    temp = temp.sort_values(
        ["intersection", "timestamp_local", "detector_id"]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # warm-up labeling
    # --------------------------------------------------------
    temp["scenario_name"] = name
    temp["scenario_date"] = rep_day
    temp["scenario_start_s"] = scenario_start_s
    temp["scenario_end_s"] = scenario_end_s
    temp["analysis_start_s"] = analysis_start_s
    temp["analysis_end_s"] = analysis_end_s
    temp["warmup_seconds"] = WARMUP_SECONDS

    temp["interval_role"] = temp["begin_s"].apply(
        lambda x: "warmup" if x < analysis_start_s else "analysis"
    )

    temp["sim_begin_s"] = temp["begin_s"] - scenario_start_s
    temp["sim_end_s"] = temp["end_s"] - scenario_start_s

    # useful labels
    temp["bin_label_clock"] = (
        temp["timestamp_local"].dt.strftime("%H:%M")
        + "-"
        + (temp["timestamp_local"] + pd.Timedelta(minutes=15)).dt.strftime("%H:%M")
    )

    temp["bin_label_sim"] = temp["sim_begin_s"].astype("Int64").astype(str) + "-" + temp["sim_end_s"].astype("Int64").astype(str)

    # --------------------------------------------------------
    # 1) FULL raw long file (includes warm-up)
    # use this for route generation / full simulation demand
    # --------------------------------------------------------
    keep_cols = [
        "scenario_name",
        "scenario_date",
        "interval_role",
        "warmup_seconds",
        "scenario_start_s",
        "scenario_end_s",
        "analysis_start_s",
        "analysis_end_s",
        "intersection",
        "date",
        "timestamp_utc",
        "timestamp_local",
        "time",
        "begin_s",
        "end_s",
        "sim_begin_s",
        "sim_end_s",
        "bin_label_clock",
        "bin_label_sim",
        "detector_id",
        "sumo_id",
        "approach",
        "edge_id",
        "movement",
        "count_type",
        "from_edge",
        "to_edge",
        "count",
        "occupancy",
        "speed",
    ]
    keep_cols_existing = [c for c in keep_cols if c in temp.columns]

    full_long_out = OUT_DIR / f"{name}_routesampler_base_long_with_warmup.csv"
    temp[keep_cols_existing].to_csv(full_long_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 2) ANALYSIS-ONLY raw long file (warm-up removed)
    # use this for validation/calibration plots and summaries
    # --------------------------------------------------------
    temp_analysis = temp[temp["interval_role"] == "analysis"].copy()

    analysis_long_out = OUT_DIR / f"{name}_routesampler_base_long_analysis_only.csv"
    temp_analysis[keep_cols_existing].to_csv(analysis_long_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # grouping columns
    # --------------------------------------------------------
    group_cols = [
        "scenario_name", "scenario_date", "interval_role",
        "intersection", "date", "begin_s", "end_s", "sim_begin_s", "sim_end_s",
        "detector_id", "sumo_id", "approach",
        "edge_id", "count_type", "from_edge", "to_edge", "movement"
    ]
    group_cols_existing = [c for c in group_cols if c in temp.columns]

    # --------------------------------------------------------
    # 3) FULL interval counts (includes warm-up)
    # --------------------------------------------------------
    detector_interval_full = (
        temp.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["begin_s", "intersection", "detector_id"])
    )

    detector_interval_full_out = OUT_DIR / f"{name}_detector_interval_counts_with_warmup.csv"
    detector_interval_full.to_csv(detector_interval_full_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 4) ANALYSIS-ONLY interval counts
    # --------------------------------------------------------
    detector_interval_analysis = (
        temp_analysis.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["begin_s", "intersection", "detector_id"])
    )

    detector_interval_analysis_out = OUT_DIR / f"{name}_detector_interval_counts_analysis_only.csv"
    detector_interval_analysis.to_csv(detector_interval_analysis_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 5) 15-min counts FULL
    # --------------------------------------------------------
    detector_15_full = (
        temp.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["intersection", "detector_id", "begin_s"])
    )

    detector_15_full_out = OUT_DIR / f"{name}_detector_15min_counts_with_warmup.csv"
    detector_15_full.to_csv(detector_15_full_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 6) 15-min counts ANALYSIS-ONLY
    # --------------------------------------------------------
    detector_15_analysis = (
        temp_analysis.groupby(group_cols_existing, dropna=False, as_index=False)
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean")
        )
        .sort_values(["intersection", "detector_id", "begin_s"])
    )

    detector_15_analysis_out = OUT_DIR / f"{name}_detector_15min_counts_analysis_only.csv"
    detector_15_analysis.to_csv(detector_15_analysis_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 7) wide matrices
    # FULL
    # --------------------------------------------------------
    pivot_index = [
        c for c in [
            "intersection", "detector_id", "sumo_id", "approach",
            "edge_id", "count_type", "from_edge", "to_edge", "movement"
        ] if c in temp.columns
    ]

    pivot_full = temp.pivot_table(
        index=pivot_index,
        columns="bin_label_clock",
        values="count",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_full_out = OUT_DIR / f"{name}_detector_time_matrix_with_warmup.csv"
    pivot_full.to_csv(pivot_full_out, index=False, encoding="utf-8-sig")

    # ANALYSIS ONLY
    pivot_analysis = temp_analysis.pivot_table(
        index=pivot_index,
        columns="bin_label_clock",
        values="count",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_analysis_out = OUT_DIR / f"{name}_detector_time_matrix_analysis_only.csv"
    pivot_analysis.to_csv(pivot_analysis_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # 8) detector summary FULL / ANALYSIS
    # --------------------------------------------------------
    summary_group_cols = [
        c for c in [
            "intersection", "detector_id", "sumo_id", "approach",
            "edge_id", "count_type", "from_edge", "to_edge", "movement"
        ] if c in temp.columns
    ]

    detector_summary_full = (
        temp.groupby(summary_group_cols, dropna=False, as_index=False)
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean"),
            n_intervals=("count", "size")
        )
        .sort_values(["intersection", "detector_id"])
    )

    detector_summary_full_out = OUT_DIR / f"{name}_detector_summary_with_warmup.csv"
    detector_summary_full.to_csv(detector_summary_full_out, index=False, encoding="utf-8-sig")

    detector_summary_analysis = (
        temp_analysis.groupby(summary_group_cols, dropna=False, as_index=False)
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_speed=("speed", "mean"),
            n_intervals=("count", "size")
        )
        .sort_values(["intersection", "detector_id"])
    )

    detector_summary_analysis_out = OUT_DIR / f"{name}_detector_summary_analysis_only.csv"
    detector_summary_analysis.to_csv(detector_summary_analysis_out, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # scenario summary
    # --------------------------------------------------------
    summary_rows.append({
        "scenario": name,
        "date": rep_day,
        "start_time_full": start_time,
        "end_time_full": end_time,
        "warmup_minutes": WARMUP_SECONDS // 60,
        "analysis_start_time": pd.to_datetime(start_time).strftime("%H:%M:%S"),
        "analysis_from_clock": f"{(scenario_start_s + WARMUP_SECONDS)//3600:02d}:{((scenario_start_s + WARMUP_SECONDS)%3600)//60:02d}:00",
        "analysis_to_clock": end_time,
        "n_rows_full": len(temp),
        "n_rows_analysis_only": len(temp_analysis),
        "n_detectors_full": temp[["intersection", "detector_id"]].drop_duplicates().shape[0] if not temp.empty else 0,
        "n_detectors_analysis_only": temp_analysis[["intersection", "detector_id"]].drop_duplicates().shape[0] if not temp_analysis.empty else 0,
        "total_count_full": temp["count"].sum(),
        "total_count_analysis_only": temp_analysis["count"].sum(),
        "mean_occupancy_full": temp["occupancy"].mean() if not temp.empty else 0,
        "mean_occupancy_analysis_only": temp_analysis["occupancy"].mean() if not temp_analysis.empty else 0
    })

# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUT_DIR / "representative_day_routesampler_base_warmup15_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n============================================================")
print("REPRESENTATIVE DAY EXTRACTION SUMMARY (15-MIN WARM-UP)")
print("============================================================")
print(summary_df.to_string(index=False))
print(f"\nSaved outputs to:\n{OUT_DIR}")


REPRESENTATIVE DAY EXTRACTION SUMMARY (15-MIN WARM-UP)
       scenario       date start_time_full end_time_full  warmup_minutes analysis_start_time analysis_from_clock analysis_to_clock  n_rows_full  n_rows_analysis_only  n_detectors_full  n_detectors_analysis_only  total_count_full  total_count_analysis_only  mean_occupancy_full  mean_occupancy_analysis_only
weekday_morning 2026-03-10        07:45:00      09:00:00              15            07:45:00            08:00:00          09:00:00          245                   196                49                         49             11632                       9295            34.314286                     34.443878
weekday_evening 2026-03-23        15:45:00      17:00:00              15            15:45:00            16:00:00          17:00:00          245                   196                49                         49             13629                      10763            42.636735                     42.387755
weekend_morning 2026-03

After identifying the representative days, the original absolute detector counts were extracted for each selected scenario. This step was necessary because the K-means clustering was based on normalized daily profiles only for pattern recognition, whereas the subsequent demand reconstruction requires absolute traffic volumes.

For each representative day, the detector data were filtered to the corresponding scenario period and aggregated by detector. In addition, the full 15-minute detector time series of the selected day was preserved for later calibration and validation. These aggregated absolute counts form the basis for the subsequent mapping of detector observations to SUMO network edges and for the generation of demand inputs for route-based traffic reconstruction.

***detector → edge → counts.xml***

In [8]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell")

LONG_FILE = BASE_DIR / "detector_data_CSV_new" / "all_intersections_common_days_long.csv"
REP_DAY_FILE = BASE_DIR / "detector_data_CSV_new" / "kmeans_representative_days" / "final_representative_day_summary.csv"

OUT_DIR = BASE_DIR / "counts_occupancy_on_edges"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {"name": "weekday_morning", "begin": 0.0, "end": 4500.0, "start_time": "07:45:00", "end_time": "09:00:00"},
    {"name": "weekday_evening", "begin": 0.0, "end": 4500.0, "start_time": "15:45:00", "end_time": "17:00:00"},
    {"name": "weekend_morning", "begin": 0.0, "end": 4500.0, "start_time": "07:45:00", "end_time": "09:00:00"},
    {"name": "weekend_evening", "begin": 0.0, "end": 4500.0, "start_time": "15:45:00", "end_time": "17:00:00"},
]

# ============================================================
# EDGE COUNT / OCCUPANCY PART
# ============================================================

EDGE_RULES = [
    # ---------------- LSA16 ----------------
    {"name": "LSA16_to_LSA10",     "members": [("LD-LSA16", 13)],                    "edge": "-202157700#5"},
    {"name": "LSA16_W_exit",       "members": [("LD-LSA16", 15)],                    "edge": "26202222#0"},
    {"name": "LSA16_S_approach",   "members": [("LD-LSA16", 1), ("LD-LSA16", 2), ("LD-LSA16", 33)], "edge": "-202157703"},
    {"name": "LSA16_N_approach",   "members": [("LD-LSA16", 6), ("LD-LSA16", 7)],   "edge": "-1182936853"},
    {"name": "LSA16_W_stopline",   "members": [("LD-LSA16", 3), ("LD-LSA16", 4)],   "edge": "-26202222#0.41"},
    {"name": "LSA16_E_approach",   "members": [("LD-LSA16", 10), ("LD-LSA16", 11)], "edge": "202157700#3"},
  

    # ---------------- LSA10 ----------------
    {"name": "LSA10_S_approach",   "members": [("LD-LSA10", 31)],                    "edge": "-1183449794"},
    {"name": "LSA10_E_approach",   "members": [("LD-LSA10", 3), ("LD-LSA10", 4)],   "edge": "75642281"},
    {"name": "LSA10_NW_approach",   "members": [("LD-LSA10", 26), ("LD-LSA10", 27)],   "edge": "-13831317#2.34"},
    #{"name": "LSA10_NW_approach",  "members": [("LD-LSA10", 2)],                     "edge": "-13831317#2"},
    {"name": "LSA10_NE_approach",  "members": [("LD-LSA10", 29)],                    "edge": "-25563346#2"},

    # ---------------- LSA1 ----------------
    {"name": "LSA1_W_approach",    "members": [("LD-LSA1", 7), ("LD-LSA1", 9)],      "edge": "-202234344"},
    {"name": "LSA1_E_approach",    "members": [("LD-LSA1", 24), ("LD-LSA1", 26), ("LD-LSA1", 14)], "edge": "-237921040"},
    {"name": "LSA1_N_approach",    "members": [("LD-LSA1", 40), ("LD-LSA1", 28), ("LD-LSA1", 30)], "edge": "237921036.41"},

    # ---------------- LSA9 ----------------
    {"name": "LSA9_W_approach",    "members": [("LD-LSA9", 1), ("LD-LSA9", 2)],      "edge": "-30329931#2"},
    {"name": "LSA9_E_approach",    "members": [("LD-LSA9", 8), ("LD-LSA9", 9)],      "edge": "202234068#0"},
    {"name": "LSA9_NE_approach",   "members": [("LD-LSA9", 6), ("LD-LSA9", 7)],      "edge": "-14811383#2"},
    {"name": "LSA9_S_approach",   "members": [("LD-LSA9", 10)],   "edge": "-14811373#0"},
]

# ============================================================
# TURN COUNT / OCCUPANCY PART
# ============================================================

TURN_RULES = [
    # ---------------- LSA16 ----------------
    {"detector": ("LD-LSA16", 4),  "from": "-26202222#0.41", "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 3),  "from": "-26202222#0.41", "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 1),  "from": "-202157703",     "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 2),  "from": "-202157703",     "to": ["893429999#1" , "-202157700#5"]},
    {"detector": ("LD-LSA16", 33), "from": "-202157703",     "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 6),  "from": "-1182936853",    "to": ["26202222#0"]},
    {"detector": ("LD-LSA16", 7),  "from": "-1182936853",    "to": ["12528053#0" , "26202222#0" ]},
    {"detector": ("LD-LSA16", 11), "from": "202157700#3",    "to": ["12528053#0"]},
    {"detector": ("LD-LSA16", 10), "from": "202157700#3",    "to": ["893429999#1", "26202222#0"]},

    # ---------------- LSA10 ----------------
    {"detector": ("LD-LSA10", 27), "from": "-13831317#2.34", "to": ["25563346#0", "-84620353#2"]},
    {"detector": ("LD-LSA10", 26), "from": "-13831317#2.34", "to": ["17707991#0", "14811377"]},
    {"detector": ("LD-LSA10", 29), "from": "-25563346#2",    "to": ["13831317#0"]},

    # ---------------- LSA1 ----------------
    #{"detector": ("LD-LSA1", 7),   "from": "-202234344",     "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 9),   "from": "-202234344",     "to": ["59617619", "-281402235#1"]},
    {"detector": ("LD-LSA1", 27),  "from": "-59617619",      "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 13),  "from": "-59617619",      "to": ["12695647"]},
    {"detector": ("LD-LSA1", 23),  "from": "-59617619",      "to": ["12695647"]},
    {"detector": ("LD-LSA1", 26),  "from": "-237921040",     "to": ["-281402232"], "via": {"-281402232": "-59617619 -281402235#1"}},
    {"detector": ("LD-LSA1", 40),  "from": "237921036.41",   "to": ["12695647"]},
    {"detector": ("LD-LSA1", 28),  "from": "237921036.41",   "to": ["-281402235#1"]},
    {"detector": ("LD-LSA1", 30),  "from": "237921036.41",   "to": ["59617619"]},

    # ---------------- LSA9 ----------------
   # ---------------- LSA9 ----------------
{"detector": ("LD-LSA9", 2), "from": "-30329931#2", "to": ["202234067", "-202234068#2"], "via": {"202234067": "-30329931#1 14811383#1","-202234068#2": "-30329931#1" }},
{"detector": ("LD-LSA9", 1), "from": "-30329931#2", "to": ["14811373#0", "-202234068#2"], "via": { "14811373#0": "-30329931#1", "-202234068#2": "-30329931#1"}},
{"detector": ("LD-LSA9", 9),   "from": "202234068#0",    "to": ["30329931#0"]},
{"detector": ("LD-LSA9", 8),   "from": "202234068#0",    "to": ["308681382#0", "202234067", "30329931#0"], "via": {"202234067": "14811383#1"}},
{"detector": ("LD-LSA9", 4),   "from": "-14811383#2",    "to": ["308681382#0", "30329931#0"]},
{"detector": ("LD-LSA9", 5),   "from": "-14811383#2",    "to": ["14811373#0", "-202234068#2"]},
]

# ============================================================
# STARTING SPLIT RATIOS
# ============================================================

TURN_SPLIT_RATIOS = {
    ("LD-LSA16", 10): [0.2, 0.8],
    ("LD-LSA16", 7): [0.7, 0.3],
    ("LD-LSA16", 2): [0.8, 0.2],
    ("LD-LSA10", 27): [0.4, 0.6],
    ("LD-LSA10", 26): [0.7, 0.3],
    ("LD-LSA1", 9):   [0.8, 0.2],
    ("LD-LSA9", 2):   [0.5, 0.5],
    ("LD-LSA9", 1):   [0.5, 0.5],
    ("LD-LSA9", 8):   [0.33, 0.34, 0.33],
    ("LD-LSA9", 4):   [0.5, 0.5],
    ("LD-LSA9", 5):   [0.5, 0.5],
}

# ============================================================
# HELPERS
# ============================================================

def prettify(elem):
    rough = ET.tostring(elem, encoding="utf-8")
    reparsed = minidom.parseString(rough)
    return reparsed.toprettyxml(indent="  ")

def split_value(total_value, n_parts, ratios=None):
    total_value = float(total_value)
    if ratios is None:
        return [total_value / n_parts] * n_parts
    if len(ratios) != n_parts:
        raise ValueError(f"Ratio count mismatch: {len(ratios)} vs {n_parts}")
    ratio_sum = sum(ratios)
    ratios = [r / ratio_sum for r in ratios]
    return [total_value * r for r in ratios]

def split_count(total_count, n_parts, ratios=None):
    total_count = float(total_count)
    if ratios is None:
        ratios = [1.0 / n_parts] * n_parts
    if len(ratios) != n_parts:
        raise ValueError(f"Ratio count mismatch: {len(ratios)} vs {n_parts}")

    ratio_sum = sum(ratios)
    ratios = [r / ratio_sum for r in ratios]

    raw = [total_count * r for r in ratios]
    ints = [int(x) for x in raw]
    remainder = int(round(total_count - sum(ints)))

    frac_order = sorted(range(n_parts), key=lambda i: raw[i] - ints[i], reverse=True)
    for i in range(remainder):
        ints[frac_order[i % n_parts]] += 1
    return ints

def build_detector_summary_for_day(df_long, rep_day, start_time, end_time):
    temp = df_long.copy()
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce").dt.strftime("%Y-%m-%d")

    if "timestamp_local" in temp.columns:
        temp["timestamp_used"] = pd.to_datetime(temp["timestamp_local"], errors="coerce")
    else:
        temp["timestamp_used"] = pd.to_datetime(temp["timestamp"], errors="coerce")

    temp["time"] = temp["timestamp_used"].dt.strftime("%H:%M:%S")

    temp = temp[temp["date"] == rep_day].copy()
    temp = temp[(temp["time"] >= start_time) & (temp["time"] < end_time)].copy()

    temp["count"] = pd.to_numeric(temp["count"], errors="coerce").fillna(0)
    temp["occupancy"] = pd.to_numeric(temp["occupancy"], errors="coerce").fillna(0)

    summary = (
        temp.groupby(["intersection", "detector_id"], as_index=False)
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            n_intervals=("count", "size")
        )
    )

    return summary

def get_detector_total(df, detector_key, value_col="total_count"):
    temp = df[(df["intersection"] == detector_key[0]) & (df["detector_id"] == detector_key[1])]
    if temp.empty or value_col not in temp.columns:
        return 0.0
    return float(pd.to_numeric(temp[value_col], errors="coerce").fillna(0).sum())

def get_detector_mean(df, detector_key, value_col="mean_occupancy"):
    temp = df[(df["intersection"] == detector_key[0]) & (df["detector_id"] == detector_key[1])]
    if temp.empty or value_col not in temp.columns:
        return 0.0
    vals = pd.to_numeric(temp[value_col], errors="coerce").dropna()
    if vals.empty:
        return 0.0
    return float(vals.mean())

# ============================================================
# LOAD INPUTS
# ============================================================

df_long = pd.read_csv(LONG_FILE)
rep_df = pd.read_csv(REP_DAY_FILE)

if "final_representative_day" not in rep_df.columns:
    raise ValueError("Column 'final_representative_day' not found in representative day summary.")

rep_day_map = dict(zip(rep_df["scenario"], rep_df["final_representative_day"]))

# ============================================================
# MAIN
# ============================================================

for sc in SCENARIOS:
    print("\n" + "=" * 70)
    print(f"SCENARIO: {sc['name']}")
    print("=" * 70)

    if sc["name"] not in rep_day_map:
        print(f"Representative day missing for scenario: {sc['name']}")
        continue

    rep_day = str(rep_day_map[sc["name"]])
    print(f"Representative day: {rep_day}")

    df = build_detector_summary_for_day(
        df_long=df_long,
        rep_day=rep_day,
        start_time=sc["start_time"],
        end_time=sc["end_time"]
    )

    if df.empty:
        print("No data found for this scenario/day/time window.")
        continue

    scenario_summary_path = OUT_DIR / f"{sc['name']}_detector_summary.csv"
    df.to_csv(scenario_summary_path, index=False, encoding="utf-8-sig")
    print("Saved:", scenario_summary_path)

    # ---------------- EDGE DATA ----------------
    edge_rows = []
    edge_debug_rows = []

    for rule in EDGE_RULES:
        total_count = 0.0
        occ_values = []

        for det_key in rule["members"]:
            det_count = get_detector_total(df, det_key, "total_count")
            det_occ = get_detector_mean(df, det_key, "mean_occupancy")

            total_count += det_count
            occ_values.append(det_occ)

            edge_debug_rows.append({
                "rule_name": rule["name"],
                "edge": rule["edge"],
                "intersection": det_key[0],
                "detector_id": det_key[1],
                "detector_total_count": det_count,
                "detector_mean_occupancy": det_occ
            })

        mean_occ = float(sum(occ_values) / len(occ_values)) if occ_values else 0.0

        edge_rows.append({
            "edge": rule["edge"],
            "count": int(round(total_count)),
            "occupancy_mean": mean_occ,
            "rule_name": rule["name"]
        })

    edge_df_raw = pd.DataFrame(edge_rows)

    edge_counts = (
        edge_df_raw.groupby("edge", as_index=False)
        .agg(
            count=("count", "sum"),
            occupancy_mean=("occupancy_mean", "mean")
        )
        .sort_values("edge")
    )

    edge_debug_df = pd.DataFrame(edge_debug_rows)

    print("\nEDGE COUNTS + OCCUPANCY")
    print(edge_counts.to_string(index=False))

    edge_root = ET.Element("data")
    edge_interval = ET.SubElement(edge_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in edge_counts.iterrows():
        ET.SubElement(edge_interval, "edge", {
            "id": str(r["edge"]),
            "entered": str(int(r["count"]))
        })

    edge_xml_path = OUT_DIR / f"edgeData_{sc['name']}.xml"
    edge_csv_path = OUT_DIR / f"edgeData_{sc['name']}.csv"
    edge_debug_path = OUT_DIR / f"edgeData_debug_{sc['name']}.csv"

    edge_counts.to_csv(edge_csv_path, index=False, encoding="utf-8-sig")
    edge_debug_df.to_csv(edge_debug_path, index=False, encoding="utf-8-sig")

    with open(edge_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(edge_root))

    print("Saved:", edge_csv_path)
    print("Saved:", edge_debug_path)
    print("Saved:", edge_xml_path)

    # ---------------- TURN DATA ----------------
    turn_rows = []

    for rule in TURN_RULES:
        det_key = rule["detector"]
        total_count = get_detector_total(df, det_key, "total_count")
        mean_occ = get_detector_mean(df, det_key, "mean_occupancy")

        if total_count <= 0 and mean_occ <= 0:
            continue

        from_edge = rule["from"]
        to_edges = rule["to"]
        via_map = rule.get("via", {})

        ratios = TURN_SPLIT_RATIOS.get(det_key)
        split_counts = split_count(total_count, len(to_edges), ratios)
        split_occs = split_value(mean_occ, len(to_edges), ratios)

        for to_edge, cnt, occ in zip(to_edges, split_counts, split_occs):
            turn_rows.append({
                "intersection": det_key[0],
                "detector_id": det_key[1],
                "from_edge": from_edge,
                "to_edge": to_edge,
                "via": via_map.get(to_edge, ""),
                "count": int(cnt),
                "occupancy_mean": float(occ)
            })

    turn_out_df = pd.DataFrame(turn_rows)

    if turn_out_df.empty:
        print("\nNo turn rows generated.")
        continue

    turn_agg = (
        turn_out_df.groupby(["from_edge", "to_edge", "via"], as_index=False)
        .agg(
            count=("count", "sum"),
            occupancy_mean=("occupancy_mean", "mean")
        )
        .sort_values(["from_edge", "to_edge", "via"])
    )

    print("\nTURN COUNTS + OCCUPANCY")
    print(turn_agg.to_string(index=False))

    turn_root = ET.Element("data")
    turn_interval = ET.SubElement(turn_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in turn_agg.iterrows():
        attrs = {
            "from": str(r["from_edge"]),
            "to": str(r["to_edge"]),
            "count": str(int(r["count"]))
        }
        if str(r["via"]).strip():
            attrs["via"] = str(r["via"]).strip()
        ET.SubElement(turn_interval, "edgeRelation", attrs)

    turn_xml_path = OUT_DIR / f"turnData_{sc['name']}.xml"
    turn_csv_path = OUT_DIR / f"turnData_{sc['name']}.csv"
    turn_debug_path = OUT_DIR / f"turnData_debug_{sc['name']}.csv"

    turn_agg.to_csv(turn_csv_path, index=False, encoding="utf-8-sig")
    turn_out_df.to_csv(turn_debug_path, index=False, encoding="utf-8-sig")

    with open(turn_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(turn_root))

    print("Saved:", turn_csv_path)
    print("Saved:", turn_debug_path)
    print("Saved:", turn_xml_path)

print("\nDONE ✅")


SCENARIO: weekday_morning
Representative day: 2026-03-10
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\weekday_morning_detector_summary.csv

EDGE COUNTS + OCCUPANCY
          edge  count  occupancy_mean
   -1182936853    203       17.600000
   -1183449794     67       44.400000
-13831317#2.34    121       43.300000
   -14811373#0     11        1.800000
   -14811383#2    223       21.100000
  -202157700#5    593       32.400000
    -202157703    476       27.400000
    -202234344    949       63.800000
    -237921040   1373       21.800000
   -25563346#2     42        9.400000
-26202222#0.41    430       46.400000
   -30329931#2    736       60.300000
   202157700#3    488       16.300000
   202234068#0    650       32.800000
  237921036.41    587       32.466667
    26202222#0    323       14.400000
      75642281    439       19.500000
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edge

***
**RandomTrips** 

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
  "C:\Program Files (x86)\Eclipse\Sumo\tools\randomTrips.py" `
  -n "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\digital_twin\simulation\landau_corridor.net.xml" `
  -b 0 `
  -e 3600 `
  -p 0.02 `
  --seed 42 `
  --weights-prefix "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\allowed_weights" `
  --fringe-factor max `
  --random-routing-factor 1.5 `
  --min-distance 50 `
  --validate `
  --remove-loops `
  -o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\randomtrips\candidate_trips.xml" `
  --route-file "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\randomtrips\candidate_routes.rou.xml"
  ***

**RouteSampler** **WEEKDAYMORNING**
& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekday_morning.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekday_morning.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
--verbose `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekday_morning_sampled_warmup15.rou.xml"


****WEEKDAY EVENING**
& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekday_evening.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekday_evening.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
--verbose `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekday_evening_sampled_warmup15.rou.xml"

**WEEKEND MORNING** 

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekend_morning.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekend_morning.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
--verbose `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekend_morning_sampled_warmup15.rou.xml"

**WEEKEND EVENING**

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
"C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
-r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml" `
--edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\edgeData_weekend_evening.xml" `
--turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\counts_occupancy_on_edges\turnData_weekend_evening.xml" `
--optimize full `
--min-count 2 `
--minimize-vehicles 1 `
--attributes 'departLane="best" departSpeed="max" type="trafficMix"' `
--verbose `
-o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\sampled\weekend_evening_sampled_warmup15.rou.xml"